# OCT Disease Classification with Vision Transformer (ViT)

**Fine-tune a ViT model to classify OCT B-scans into 4 categories.**

Categories:
- **CNV** — Choroidal Neovascularization (wet AMD)
- **DME** — Diabetic Macular Edema
- **DRUSEN** — Drusen deposits (dry AMD)
- **NORMAL** — No abnormalities

**Dataset:** Kermany OCT (84,484 images)
**Model:** google/vit-base-patch16-224
**Target:** 99%+ test accuracy
**Runtime:** ~30 min on T4 GPU (Colab free tier)

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q transformers datasets torch torchvision Pillow scikit-learn \
    matplotlib seaborn pandas kagglehub huggingface-hub accelerate tqdm

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import transforms
from transformers import (
    ViTForImageClassification,
    TrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
    Trainer,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import label_binarize

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Download Kermany OCT Dataset

~84K OCT B-scan images across 4 classes. Pre-split into train/test.

In [ ]:
# Set your Kaggle credentials (or upload kaggle.json to ~/.kaggle/)
os.environ["KAGGLE_USERNAME"] = "eliasteikari"
os.environ["KAGGLE_KEY"] = ""

import kagglehub

# Download Kermany OCT dataset
data_path = kagglehub.dataset_download("paultimothymooney/kermany2018")
print(f"Dataset downloaded to: {data_path}")

# Find the OCT2017 directory
oct_dir = os.path.join(data_path, "OCT2017")
if not os.path.isdir(oct_dir):
    # Try data_path directly
    oct_dir = data_path

for item in sorted(os.listdir(oct_dir)):
    full = os.path.join(oct_dir, item)
    if os.path.isdir(full):
        n_files = sum(len(files) for _, _, files in os.walk(full))
        print(f"  {item}/ ({n_files} files)")
    else:
        size = os.path.getsize(full)
        print(f"  {item} ({size})")

## 3. Load & Explore the Dataset

In [ ]:
# Constants
OCT_CLASSES = ["CNV", "DME", "DRUSEN", "NORMAL"]
NUM_CLASSES = len(OCT_CLASSES)

# Load dataset from folder structure
def load_split(split_dir):
    """Load all images from a train/ or test/ directory."""
    records = []
    for class_idx, class_name in enumerate(OCT_CLASSES):
        class_dir = os.path.join(split_dir, class_name)
        if not os.path.isdir(class_dir):
            print(f"Warning: {class_dir} not found")
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith((".jpeg", ".jpg", ".png")):
                records.append({
                    "image_path": os.path.join(class_dir, fname),
                    "label": class_idx,
                    "label_name": class_name,
                })
    return pd.DataFrame(records)

train_full_df = load_split(os.path.join(oct_dir, "train"))
test_df = load_split(os.path.join(oct_dir, "test"))

print(f"Train: {len(train_full_df)} images")
print(f"Test:  {len(test_df)} images")
print(f"\nTrain class distribution:")
print(train_full_df["label_name"].value_counts())
print(f"\nTest class distribution:")
print(test_df["label_name"].value_counts())

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, df) in zip(axes, [("Train", train_full_df), ("Test", test_df)]):
    counts = df["label_name"].value_counts().reindex(OCT_CLASSES)
    colors = sns.color_palette("husl", NUM_CLASSES)
    counts.plot(kind="bar", ax=ax, color=colors)
    ax.set_title(f"{title} Set Distribution ({len(df)} images)", fontsize=13)
    ax.set_ylabel("Number of Images")
    ax.set_xlabel("OCT Category")
    plt.sca(ax)
    plt.xticks(rotation=0)
    for i, v in enumerate(counts.values):
        ax.text(i, v + max(counts) * 0.01, str(v), ha="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Show sample images from each class
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (cls_name, ax) in enumerate(zip(OCT_CLASSES, axes.flat)):
    subset = train_full_df[train_full_df["label_name"] == cls_name]
    if len(subset) > 0:
        sample = subset.sample(1, random_state=42).iloc[0]
        img = Image.open(sample["image_path"]).convert("RGB")
        ax.imshow(img)
        ax.set_title(cls_name, fontsize=13, fontweight="bold")
    ax.axis("off")
plt.suptitle("Sample OCT B-Scans per Category", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Prepare Data

In [ ]:
# Hyperparameters
IMAGE_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 10
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
VAL_SPLIT = 0.10
SEED = 42
PATIENCE = 3

# OCT-specific transforms
# Key differences from fundus: no vertical flip (fixed orientation), no hue/sat jitter (grayscale)
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
class OCTDataset(Dataset):
    """Dataset returning dicts compatible with HuggingFace Trainer."""
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        label = row["label"]
        if self.transform:
            image = self.transform(image)
        return {"pixel_values": image, "labels": label}

# Carve validation set from training data (original val set is too small)
train_df, val_df = train_test_split(
    train_full_df, test_size=VAL_SPLIT, stratify=train_full_df["label"], random_state=SEED
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Create datasets
train_dataset = OCTDataset(train_df, transform=train_transforms)
val_dataset = OCTDataset(val_df, transform=val_transforms)
test_dataset = OCTDataset(test_df, transform=val_transforms)

# Class weights for imbalanced data
train_labels = train_df["label"].values
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float32)
class_counts = np.maximum(class_counts, 1.0)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights = torch.FloatTensor(class_weights)
print(f"\nClass weights: {dict(zip(OCT_CLASSES, class_weights.numpy().round(3)))}")

## 5. Model & Trainer Setup

In [ ]:
# Load pretrained ViT with new 4-class head
model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
)

# Set label mapping in model config (used by push_to_hub)
model.config.id2label = {i: name for i, name in enumerate(OCT_CLASSES)}
model.config.label2id = {name: i for i, name in enumerate(OCT_CLASSES)}

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")

In [ ]:
# Custom Trainer with class-weighted loss
class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        else:
            loss_fn = nn.CrossEntropyLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# History callback
class HistoryCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
        self._current_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        if "loss" in logs and "eval_loss" not in logs:
            self._current_train_loss = logs["loss"]
        if "eval_loss" in logs:
            self.history["val_loss"].append(logs["eval_loss"])
            self.history["val_acc"].append(logs.get("eval_accuracy", 0.0))
            if self._current_train_loss is not None:
                self.history["train_loss"].append(self._current_train_loss)
                self.history["train_acc"].append(0.0)
                self._current_train_loss = None

    def on_train_end(self, args, state, control, **kwargs):
        os.makedirs(self.output_dir, exist_ok=True)
        with open(os.path.join(self.output_dir, "history.json"), "w") as f:
            json.dump(self.history, f, indent=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {"accuracy": (predictions == labels).mean()}

## 6. Train with HuggingFace Trainer

In [ ]:
OUTPUT_DIR = "oct_disease_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_total_limit=2,
    fp16=(device.type == "cuda"),
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    report_to="none",
)

history_callback = HistoryCallback(OUTPUT_DIR)

trainer = WeightedLossTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[history_callback, EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

In [ ]:
# Train!
trainer.train()

# Save best model
best_model_dir = os.path.join(OUTPUT_DIR, "best_model")
trainer.save_model(best_model_dir)

# Save label mapping
label_map = {i: name for i, name in enumerate(OCT_CLASSES)}
with open(os.path.join(best_model_dir, "label_map.json"), "w") as f:
    json.dump(label_map, f, indent=2)

history = history_callback.history
best_val_acc = max(history["val_acc"]) if history["val_acc"] else 0.0
print(f"\nTraining complete! Best validation accuracy: {100*best_val_acc:.2f}%")

## 7. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_range, history["train_loss"], "b-o", label="Train", markersize=4)
ax1.plot(epochs_range, history["val_loss"], "r-o", label="Validation", markersize=4)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, [a*100 for a in history["val_acc"]], "r-o", label="Validation", markersize=4)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Validation Accuracy")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=150)
plt.show()

## 8. Evaluate on Test Set

In [ ]:
# Load best model
best_model = ViTForImageClassification.from_pretrained(os.path.join(OUTPUT_DIR, "best_model"))
best_model = best_model.to(device)
best_model.eval()

# Get predictions using Trainer
test_results = trainer.predict(test_dataset)
y_probs = torch.softmax(torch.tensor(test_results.predictions), dim=1).numpy()
y_pred = np.argmax(test_results.predictions, axis=1)
y_true = test_results.label_ids

# Classification report
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=OCT_CLASSES, digits=4))

# Overall accuracy
accuracy = (y_pred == y_true).mean()
print(f"Overall Test Accuracy: {100*accuracy:.2f}%")

In [ ]:
# Per-class AUC
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
print("Per-class AUC (One-vs-Rest):")
auc_scores = {}
for i, name in enumerate(OCT_CLASSES):
    if y_true_bin[:, i].sum() > 0:
        auc = roc_auc_score(y_true_bin[:, i], y_probs[:, i])
        auc_scores[name] = auc
        print(f"  {name:10s}: {auc:.4f}")

mean_auc = np.mean(list(auc_scores.values()))
print(f"  {'Mean AUC':10s}: {mean_auc:.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=OCT_CLASSES, yticklabels=OCT_CLASSES)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("True", fontsize=12)
plt.title("Confusion Matrix — OCT Classification", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

## 9. Try It Out — Predict on Sample Images

In [ ]:
def predict_and_show(image_path, model, device):
    """Predict disease and display the image with results."""
    image = Image.open(image_path).convert("RGB")
    tensor = val_transforms(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(tensor).logits
        probs = torch.softmax(outputs, dim=1).squeeze().cpu().numpy()

    predicted_idx = probs.argmax()
    predicted_class = OCT_CLASSES[predicted_idx]
    confidence = probs[predicted_idx]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.imshow(image)
    ax1.set_title(f"Prediction: {predicted_class} ({confidence*100:.1f}%)", fontsize=14, fontweight="bold")
    ax1.axis("off")

    colors = ["green" if i == predicted_idx else "steelblue" for i in range(NUM_CLASSES)]
    bars = ax2.barh(OCT_CLASSES, probs * 100, color=colors)
    ax2.set_xlabel("Confidence (%)")
    ax2.set_title("Class Probabilities")
    ax2.set_xlim(0, 100)
    for bar, prob in zip(bars, probs):
        if prob > 0.02:
            ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                     f"{prob*100:.1f}%", va="center", fontsize=10)

    plt.tight_layout()
    plt.show()

# Test on random samples from the test set
test_samples = test_df.sample(4, random_state=42)
for _, row in test_samples.iterrows():
    print(f"\nTrue label: {row['label_name']}")
    predict_and_show(row["image_path"], best_model, device)

## 10. Save & Push to HuggingFace Hub

In [ ]:
# Show saved model files
print(f"Model saved at: {os.path.join(OUTPUT_DIR, 'best_model')}")
print(f"\nFiles:")
model_dir = os.path.join(OUTPUT_DIR, "best_model")
for f in sorted(os.listdir(model_dir)):
    size_mb = os.path.getsize(os.path.join(model_dir, f)) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

In [ ]:
# Push to HuggingFace Hub
# Uncomment and fill in your details:

# from huggingface_hub import login
# login()  # Will prompt for your HF token
#
# trainer.push_to_hub("eliasteikari/oct-disease-vit")
# print("Model pushed to HuggingFace Hub!")

## Done!

You now have a trained OCT disease classifier. Expected test accuracy: **99%+**

Next steps:

1. **Push to HuggingFace Hub** using `trainer.push_to_hub()` above
2. **Deploy to HuggingFace Spaces** — the Gradio app auto-detects OCT models
3. **Run the Gradio app** locally: `python app/gradio_app.py --model_dir checkpoints_oct/best_model`
4. **Show your dad!** Upload OCT scans and get instant classifications

**Disclaimer:** This is a screening/educational tool, not a diagnostic device. Always consult a qualified ophthalmologist.